# Numbeo Crime: Data Exploration

Each exploration notebook for a data source should work through the key steps:
1. Loading the data
2. Determining the observation units and the variables of interest
3. Locating and handling missing data
4. Transforming/adding new variables
5. Creating and saving clean subsets
6. Identifying the key dimensions in the dataset (temporal, spatial, categorical)
7. Listing the key questions relating to the overarching research questions
8. Carrying out descriptive analysis of each of the key variables and relevant combinations

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import math
import plotly
import warnings
warnings.simplefilter('ignore')
import numpy as np
from scipy import stats

def load_numbeo_file(filepath):
    """Load all year-sheets from a Numbeo Excel file into one DataFrame."""
    all_years = []
    xl = pd.ExcelFile(filepath)
    for sheet in xl.sheet_names:
        try:
            df = pd.read_excel(filepath, sheet_name=sheet)
            df['year'] = int(str(sheet).strip())
            all_years.append(df)
            print(f'  Loaded {sheet}: {len(df)} rows')
        except Exception as e:
            print(f'  Skipped {sheet}: {e}')
    return pd.concat(all_years, ignore_index=True)

## 1. Loading the Data

Numbeo publishes two crime datasets:
- **`numbeo_crime.xlsx`** — city-level Crime Index and Safety Index, one sheet per year
- **`numbeo_crime_country.xlsx`** — country-level aggregates, one sheet per year

Both indices are Numbeo's composite scores derived from user surveys. The **Crime Index** is higher for more dangerous places (0–100+); the **Safety Index** is its complement (100 − Crime Index approximately).

In [ ]:
print('Loading city data...')
city_raw = load_numbeo_file('../data/raw/numbeo_crime.xlsx')

print('\nLoading country data...')
country_raw = load_numbeo_file('../data/raw/numbeo_crime_country.xlsx')

print(f'\nCity shape:    {city_raw.shape}')
print(f'Country shape: {country_raw.shape}')
city_raw.head(4)

## 2. Observation Units and Variables of Interest

**City dataset — observation unit:** City × Year

**Country dataset — observation unit:** Country × Year

| Variable | Description |
|---|---|
| `City` / `Country` | Name (city includes `City, Country` format) |
| `Crime Index` | Composite crime score (higher = more crime) |
| `Safety Index` | Composite safety score (higher = safer) |
| `year` | Survey year (2016–2026 for cities; 2016–2025 for countries) |

City names follow the format `"City, Country"` — we parse the country from this for regional grouping.

In [ ]:
print('City dataset:')
print(f'  Years: {sorted(city_raw["year"].unique())}')
print(f'  Unique cities: {city_raw["City"].nunique()}')
print(f'  Columns: {city_raw.columns.tolist()}')

print('\nCountry dataset:')
print(f'  Years: {sorted(country_raw["year"].unique())}')
print(f'  Unique countries: {country_raw["Country"].nunique()}')

## 3. Locating and Handling Missing Data

In [ ]:
print('=== City missing values ===')
print(city_raw.isna().sum())

print('\n=== Country missing values ===')
print(country_raw.isna().sum())

# Check coverage: how many cities appear in every year?
city_year_counts = city_raw.groupby('City')['year'].count()
max_years = city_raw['year'].nunique()
print(f'\nMax possible years per city: {max_years}')
print(f'Cities present in ALL years: {(city_year_counts == max_years).sum()}')
print(f'Cities present in only 1 year: {(city_year_counts == 1).sum()}')

The raw data has no NaN values — Numbeo only publishes scores when it has sufficient survey responses, so every row is complete. However, coverage is uneven across years: many cities appear in some years but not others. This is a structural gap rather than missing data — we handle it by working with the most recent year for cross-sectional analysis and using only consistently covered cities for trend analysis.

## 4. Transforming and Adding New Variables

In [ ]:
city = city_raw.copy()

# Parse country from 'City, Country' format
city['country'] = city['City'].str.split(',').str[-1].str.strip()
city['city_name'] = city['City'].str.split(',').str[0].str.strip()

# Rename columns to snake_case
city = city.rename(columns={
    'City': 'city_full',
    'Crime Index': 'crime_index',
    'Safety Index': 'safety_index',
    'Rank': 'rank'
})

# Safety tier for easy filtering
def safety_tier(score):
    if score >= 70:   return 'Very Safe'
    elif score >= 55: return 'Safe'
    elif score >= 40: return 'Moderate'
    elif score >= 25: return 'Unsafe'
    else:             return 'Very Unsafe'

city['safety_tier'] = city['safety_index'].apply(safety_tier)

print(city[['city_name', 'country', 'crime_index', 'safety_index', 'safety_tier', 'year']].head(6))

In [ ]:
country = country_raw.copy()
country = country.rename(columns={
    'Country': 'country',
    'Crime Index': 'crime_index',
    'Safety Index': 'safety_index',
    'Rank': 'rank'
})
country['safety_tier'] = country['safety_index'].apply(safety_tier)

# Year-over-year change in crime index per country
country_sorted = country.sort_values(['country', 'year'])
country_sorted['crime_yoy'] = country_sorted.groupby('country')['crime_index'].diff()

print(country.head(4))

## 5. Creating and Saving Clean Subsets

In [ ]:
import os
os.makedirs('../data/clean', exist_ok=True)

# Full city and country datasets
city.to_csv('../data/clean/numbeo_crime_city.csv', index=False)
country_sorted.to_csv('../data/clean/numbeo_crime_country.csv', index=False)
print(f'Saved numbeo_crime_city.csv    — {city.shape[0]} rows')
print(f'Saved numbeo_crime_country.csv — {country_sorted.shape[0]} rows')

# Latest year only
city_latest    = city[city['year'] == city['year'].max()].copy()
country_latest = country[country['year'] == country['year'].max()].copy()
city_latest.to_csv('../data/clean/numbeo_crime_city_latest.csv', index=False)
country_latest.to_csv('../data/clean/numbeo_crime_country_latest.csv', index=False)
print(f'Saved numbeo_crime_city_latest.csv    — {city_latest.shape[0]} rows ({city["year"].max()})')
print(f'Saved numbeo_crime_country_latest.csv — {country_latest.shape[0]} rows ({country["year"].max()})')

## 6. Key Dimensions in the Dataset

- **Temporal:** 2016–2026 (cities), 2016–2025 (countries), annual Numbeo surveys
- **Spatial:** ~507 unique cities; ~130 unique countries
- **Categorical:** Safety tier (Very Safe → Very Unsafe)
- **Key limitation:** Numbeo scores are crowd-sourced perception data, not official crime statistics. They capture *felt* safety rather than objective crime rates, and are biased toward cities with more Numbeo users (typically wealthier, more internet-connected populations).

In [ ]:
print('Cities per year:')
print(city.groupby('year')['city_name'].count().to_string())

print('\nCountries per year:')
print(country.groupby('year')['country'].count().to_string())

## 7. Key Questions Relating to the Research Questions

The project asks: **which cities offer the best value for living** — high quality of life at reasonable cost? Safety is a core quality-of-life dimension.

1. **Which cities are the safest in the most recent year?** — Direct input to a city quality-of-life score.
2. **Which countries have the lowest crime index?** — Country-level safety context for evaluating cities.
3. **How has city-level safety changed over time?** — Are cities getting safer or more dangerous?
4. **What is the distribution of safety scores globally?** — Is safety bimodal (safe vs. unsafe) or continuous?
5. **Which regions of the world are safest?** — Geographic clustering of safety.
6. **Are there cities that are both affordable (low cost of living) and safe?** — The key value-for-living intersection.

## 8. Descriptive Analysis

### Q1: Which cities are safest in the most recent year?

In [ ]:
latest_year = city['year'].max()
cl = city[city['year'] == latest_year].copy()

print(f'Crime/Safety Index summary — cities ({latest_year}):')
print(cl[['crime_index','safety_index']].describe().round(2))

print(f'\nTop 20 safest cities ({latest_year}):')
print(cl.nlargest(20,'safety_index')[['city_name','country','safety_index','crime_index']]
      .to_string(index=False))

print(f'\nTop 20 most dangerous cities ({latest_year}):')
print(cl.nlargest(20,'crime_index')[['city_name','country','crime_index','safety_index']]
      .to_string(index=False))

In [ ]:
safe20 = cl.nlargest(20, 'safety_index').sort_values('safety_index')
danger20 = cl.nlargest(20, 'crime_index').sort_values('crime_index')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

axes[0].barh(safe20['city_name'] + ', ' + safe20['country'],
             safe20['safety_index'], color='#059669')
axes[0].set_title(f'Top 20 Safest Cities — Safety Index ({latest_year})', fontsize=12)
axes[0].set_xlabel('Safety Index (higher = safer)')
axes[0].set_xlim(60, 100)

axes[1].barh(danger20['city_name'] + ', ' + danger20['country'],
             danger20['crime_index'], color='#DC2626')
axes[1].set_title(f'Top 20 Most Dangerous Cities — Crime Index ({latest_year})', fontsize=12)
axes[1].set_xlabel('Crime Index (higher = more crime)')

plt.tight_layout()
plt.show()

### Q2: Which countries have the lowest crime index?

In [ ]:
country_latest_yr = country['year'].max()
cntry = country[country['year'] == country_latest_yr].copy()

print(f'Safest countries ({country_latest_yr}):')
print(cntry.nsmallest(20,'crime_index')[['country','crime_index','safety_index']]
      .to_string(index=False))

print(f'\nMost dangerous countries ({country_latest_yr}):')
print(cntry.nlargest(20,'crime_index')[['country','crime_index','safety_index']]
      .to_string(index=False))

In [ ]:
# Need ISO3 codes for choropleth — merge via pycountry or manual mapping
# Use country name directly; plotly can resolve many names
fig = px.choropleth(
    cntry,
    locations='country',
    locationmode='country names',
    color='crime_index',
    hover_name='country',
    color_continuous_scale='RdYlGn_r',
    title=f'Crime Index by Country ({country_latest_yr}) — darker = more crime',
    labels={'crime_index': 'Crime Index'}
)
fig.update_layout(geo=dict(showframe=False, showcoastlines=True))
fig.show()

### Q3: How has city-level safety changed over time?

We track cities that appear consistently across most years to measure genuine trends rather than sampling changes.

In [ ]:
# Cities present in at least 8 of the available years
min_years = 8
consistent_cities = city_year_counts[city_year_counts >= min_years].index
city_consistent = city[city['city_full'].isin(consistent_cities)].copy()
print(f'Cities with {min_years}+ years of data: {len(consistent_cities)}')

# Global average safety trend
global_trend = city.groupby('year')[['crime_index','safety_index']].mean().reset_index()
print('\nGlobal average safety index by year:')
print(global_trend.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(global_trend['year'], global_trend['safety_index'], marker='o', color='#059669', linewidth=2)
ax.fill_between(global_trend['year'], global_trend['safety_index'], alpha=0.1, color='#059669')
ax.set_title('Global Average Safety Index Across All Cities (2016–present)', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Average Safety Index')
ax.set_xticks(global_trend['year'])
plt.tight_layout()
plt.show()

In [ ]:
# Spotlight: cities with biggest safety improvement and biggest decline
first_last = (
    city_consistent
    .groupby('city_full')
    .apply(lambda g: pd.Series({
        'safety_first': g.loc[g['year'].idxmin(), 'safety_index'],
        'safety_last':  g.loc[g['year'].idxmax(), 'safety_index'],
        'year_first': g['year'].min(),
        'year_last':  g['year'].max(),
        'country': g['country'].iloc[0]
    }))
    .reset_index()
)
first_last['change'] = first_last['safety_last'] - first_last['safety_first']

print('Top 10 most improved cities (safety):')
print(first_last.nlargest(10,'change')[['city_full','country','safety_first','safety_last','change']]
      .to_string(index=False))

print('\nTop 10 most deteriorated cities (safety):')
print(first_last.nsmallest(10,'change')[['city_full','country','safety_first','safety_last','change']]
      .to_string(index=False))

### Q4: Distribution of safety scores globally

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(cl['safety_index'], bins=30, color='#3B82F6', edgecolor='white')
axes[0].axvline(cl['safety_index'].mean(), color='red', linestyle='--',
                label=f'Mean = {cl["safety_index"].mean():.1f}')
axes[0].set_title(f'Distribution of Safety Index — Cities ({latest_year})')
axes[0].set_xlabel('Safety Index')
axes[0].set_ylabel('Number of Cities')
axes[0].legend()

tier_counts = cl['safety_tier'].value_counts().reindex(
    ['Very Safe', 'Safe', 'Moderate', 'Unsafe', 'Very Unsafe']
).fillna(0)
colors = ['#059669', '#34D399', '#FBBF24', '#EF4444', '#991B1B']
axes[1].bar(tier_counts.index, tier_counts.values, color=colors)
axes[1].set_title(f'Cities by Safety Tier ({latest_year})')
axes[1].set_xlabel('Safety Tier')
axes[1].set_ylabel('Number of Cities')

plt.tight_layout()
plt.show()

print('\nSafety tier breakdown:')
pct = (tier_counts / tier_counts.sum() * 100).round(1)
for tier, n, p in zip(tier_counts.index, tier_counts.values, pct.values):
    print(f'  {tier:<15} {int(n):>4} cities  ({p}%)')

The safety index distribution is roughly bimodal — cities cluster either in the "safe" range (50–80) or the "unsafe" range (10–40), with relatively fewer cities in the middle. This reflects Numbeo's coverage: it skews toward either wealthy cities in developed nations (safer) or major cities in Latin America and Africa (less safe).

### Q5: Which regions of the world are safest?

In [ ]:
# Broad regional grouping based on country
region_map = {
    'Europe': ['Germany','France','Spain','Italy','Netherlands','Belgium','Sweden','Norway',
               'Denmark','Finland','Austria','Switzerland','Portugal','Poland','Czechia',
               'Hungary','Romania','Greece','Croatia','Slovakia','Slovenia','Estonia',
               'Latvia','Lithuania','Luxembourg','Ireland','Iceland','Serbia','Ukraine',
               'Bulgaria','Bosnia and Herzegovina','Albania','North Macedonia','Kosovo',
               'Montenegro','Moldova','Belarus','Russia','Turkey'],
    'East Asia & Pacific': ['Japan','China','South Korea','Australia','New Zealand',
                            'Singapore','Taiwan','Hong Kong','Thailand','Vietnam',
                            'Malaysia','Indonesia','Philippines','India','Sri Lanka',
                            'Bangladesh','Pakistan','Nepal'],
    'North America': ['United States','Canada','Mexico'],
    'Latin America': ['Brazil','Colombia','Argentina','Chile','Peru','Venezuela','Ecuador',
                      'Bolivia','Paraguay','Uruguay','Honduras','Guatemala','El Salvador',
                      'Costa Rica','Panama','Dominican Republic','Cuba','Puerto Rico'],
    'Middle East & Africa': ['South Africa','Egypt','Nigeria','Kenya','Morocco','Ghana',
                             'Ethiopia','Tanzania','Uganda','Cameroon','Israel','UAE',
                             'Saudi Arabia','Qatar','Kuwait','Jordan','Lebanon','Iran','Iraq']
}

country_to_region = {c: r for r, countries in region_map.items() for c in countries}
cl['region'] = cl['country'].map(country_to_region).fillna('Other')

region_stats = cl.groupby('region')[['safety_index','crime_index']].agg(['mean','median','count']).round(2)
print('Safety by region (latest year):')
print(region_stats.to_string())

In [ ]:
region_order = cl.groupby('region')['safety_index'].median().sort_values(ascending=False).index

fig = px.box(
    cl[cl['region'] != 'Other'],
    x='region',
    y='safety_index',
    color='region',
    category_orders={'region': list(region_order)},
    points='all',
    hover_name='city_name',
    title=f'Safety Index by Region — Cities ({latest_year})',
    labels={'safety_index': 'Safety Index', 'region': 'Region'}
)
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white', showlegend=False,
                  xaxis_tickangle=-20)
fig.show()

Europe and East Asia & Pacific have the highest and most consistent safety scores. Latin America has the widest spread — some cities (e.g. Santiago, Montevideo) are quite safe while others are among the most dangerous globally. North America also shows high variance, driven by US cities ranging from very safe to quite dangerous.

### Q6: Are there cities that are both affordable and safe?

We cross the safety index with the Numbeo Cost of Living Index to find the value sweet spots: cities where safety is high but cost is moderate or low.

In [ ]:
import os

col_path = '../data/clean/numbeo_col_city.csv'

if os.path.exists(col_path):
    col = pd.read_csv(col_path)
    # Align to same latest year
    col_yr = col[col['year'] == col['year'].max()].copy()

    # Standardise city name for merge
    col_yr['city_full_clean'] = col_yr['city_full'].str.strip().str.lower()
    cl['city_full_clean'] = cl['city_full'].str.strip().str.lower()

    merged = pd.merge(
        cl[['city_name','country','city_full_clean','safety_index','crime_index','region']],
        col_yr[['city_full_clean','cost_of_living_index']],
        on='city_full_clean',
        how='inner'
    ).dropna(subset=['cost_of_living_index','safety_index'])

    print(f'Cities with both safety + cost data: {len(merged)}')

    r, p = stats.pearsonr(merged['cost_of_living_index'], merged['safety_index'])
    print(f'Correlation (cost vs safety): r = {r:.3f}, p = {p:.4f}')

    fig = px.scatter(
        merged,
        x='cost_of_living_index',
        y='safety_index',
        hover_name='city_name',
        color='region',
        title=f'Safety vs Cost of Living — Cities (r = {r:.3f})',
        labels={'cost_of_living_index': 'Cost of Living Index (higher = more expensive)',
                'safety_index': 'Safety Index (higher = safer)'}
    )
    # Quadrant lines at medians
    fig.add_hline(y=merged['safety_index'].median(), line_dash='dash', line_color='grey')
    fig.add_vline(x=merged['cost_of_living_index'].median(), line_dash='dash', line_color='grey')
    fig.update_layout(plot_bgcolor='white', paper_bgcolor='white')
    fig.show()

    # Sweet spot: top-right = high safety, low cost
    med_cost   = merged['cost_of_living_index'].median()
    med_safety = merged['safety_index'].median()
    sweet = merged[(merged['safety_index'] > med_safety) &
                   (merged['cost_of_living_index'] < med_cost)]
    print(f'\nSweet-spot cities (safe + affordable): {len(sweet)}')
    print(sweet.nlargest(20,'safety_index')[['city_name','country','safety_index','cost_of_living_index']]
          .to_string(index=False))
else:
    print('Cost of living clean data not yet available.')
    print('Run 03_numbeo_cost_of_living_exploration_city.ipynb first to generate it.')
    print('\nShowing safety-only ranking for now:')
    print(cl.nlargest(20,'safety_index')[['city_name','country','safety_index','crime_index']]
          .to_string(index=False))

### Country-Level Crime Trends Over Time

In [ ]:
# Track a selection of notable countries
spotlight = ['Japan', 'Germany', 'United States', 'Brazil', 'South Africa',
             'Australia', 'Singapore', 'Colombia', 'Spain', 'India']

spot_trend = country[country['country'].isin(spotlight)].copy()

fig = px.line(
    spot_trend,
    x='year',
    y='safety_index',
    color='country',
    markers=True,
    title='Safety Index Over Time — Selected Countries (2016–present)',
    labels={'safety_index': 'Safety Index', 'year': 'Year'}
)
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Countries with biggest average improvement in safety over the full period
country_change = (
    country.groupby('country')
    .apply(lambda g: g.loc[g['year'].idxmax(), 'safety_index'] -
                     g.loc[g['year'].idxmin(), 'safety_index']
           if len(g) >= 4 else np.nan)
    .dropna()
    .sort_values(ascending=False)
    .rename('safety_change')
    .reset_index()
)

print('Top 15 countries — biggest safety improvement:')
print(country_change.head(15).to_string(index=False))

print('\nTop 15 countries — biggest safety decline:')
print(country_change.tail(15).sort_values('safety_change').to_string(index=False))

## Summary

| Finding | Implication for Value-for-Living Analysis |
|---|---|
| Europe and East Asia dominate the safest city rankings | Key regions to focus on for high-quality affordable living |
| Latin America concentrates the most dangerous cities | Cities here unlikely to score well on quality-adjusted value |
| Safety distribution is roughly bimodal | Few "middle-ground" cities — places tend to be either clearly safe or clearly unsafe |
| Global average safety has been broadly stable 2016–present | No systemic global improvement or deterioration |
| Scores are perception-based | Should be cross-validated against official crime statistics where possible |
| Sweet-spot analysis (safe + affordable) | Depends on cost-of-living data from notebook 03 — run that first |